In [12]:
import polars as pl
from random import randint
from time import perf_counter_ns


def create_expression(current_depth=0, max_depth=10, rounds=10):
    if current_depth >= max_depth:
        return pl.struct(
            pl.lit(randint(0, 10)).alias("a"),
            pl.lit(randint(0, 10)).alias("b"),
            pl.lit(randint(0, 10)).alias("c"),
        )

    expr = pl
    for r in range(rounds):
        expr = expr.when(pl.col(f"feat_{current_depth}") < r).then(
            create_expression(current_depth + 1, max_depth, rounds)
        )
    expr = expr.otherwise(create_expression(current_depth + 1, max_depth, rounds))
    return expr


expr = create_expression(max_depth=5, rounds=8)

df = pl.DataFrame({f"feat_{i}": [randint(0, 10) for _ in range(10)] for i in range(10)})

# We want to speedup this bit
t1 = perf_counter_ns()
df.select(expr.alias("result"))
total = perf_counter_ns() - t1
print(f"Execution time: {total / 1e6:.2f} ms")

Execution time: 5219.37 ms


In [17]:
res_df = []

def create_expression(current_depth=0, max_depth=10, rounds=10):
    if current_depth >= max_depth:
        idx = len(res_df)
        res_df.append({"idx": idx, "a": randint(0, 10), "b": randint(0, 10), "c": randint(0, 10)})
        return pl.lit(idx)

    expr = pl
    for r in range(rounds):
        expr = expr.when(pl.col(f"feat_{current_depth}") < r).then(
            create_expression(current_depth + 1, max_depth, rounds)
        )
    expr = expr.otherwise(create_expression(current_depth + 1, max_depth, rounds))
    return expr

expr = create_expression(max_depth=5, rounds=8)
res_df = pl.DataFrame(res_df)

t1 = perf_counter_ns()
df.select(expr.alias("result"))
total = perf_counter_ns() - t1
print(f"Execution time: {total / 1e6:.2f} ms")

Execution time: 754.38 ms


result,a,b,c
i32,i64,i64,i64
8216,5,7,10
9470,6,3,6
11906,4,1,0
11969,6,7,0
11990,9,7,9
15252,5,10,1
16120,1,6,3
16588,8,8,0
16659,5,6,6
